### Importar librerías cirq y qsimcirq

In [1]:
try:
    import cirq
except ImportError:
    !pip install cirq --quiet
    import cirq

try:
    import qsimcirq
except ImportError:
    !pip install qsimcirq --quiet
    import qsimcirq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 430.5/430.5 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.3/580.3 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 24.4 MB/s eta 0:00:00


In [2]:
import cirq
import numpy as np
from scipy.optimize import minimize

n = 12

J = 1.0
h = 0.5


p = 6

In [6]:
def build_qaoa_circuit(gammas, betas):
    qubits = cirq.LineQubit.range(n)
    circuit = cirq.Circuit()

    # Estado inicial |+>
    circuit.append(cirq.H.on_each(*qubits))

    for layer in range(p):
        gamma = gammas[layer]
        beta = betas[layer]

        # --- Cost Hamiltonian (Ising 1D) ---
        for i in range(n - 1):
            circuit.append(cirq.ZZ(qubits[i], qubits[i+1]) ** (gamma * J / np.pi))

        for i in range(n):
            circuit.append(cirq.Z(qubits[i]) ** (gamma * h / np.pi))

        # --- Mixer ---
        for i in range(n):
            circuit.append(cirq.X(qubits[i]) ** (2*beta / np.pi))

    return circuit

In [9]:
sim = qsimcirq.QSimSimulator()

def compute_energy(state_vector):
    energy = 0.0

    # ZZ terms
    for i in range(n - 1):
        for basis_state, amp in enumerate(state_vector):
            prob = np.abs(amp)**2

            zi = 1 if ((basis_state >> i) & 1) == 0 else -1
            zj = 1 if ((basis_state >> (i+1)) & 1) == 0 else -1

            energy +=-J * zi * zj * prob

    # Z terms
    for i in range(n):
        for basis_state, amp in enumerate(state_vector):
            prob = np.abs(amp)**2

            zi = 1 if ((basis_state >> i) & 1) == 0 else -1

            energy += -h * zi * prob

    return energy

In [7]:
def objective(params):
    gammas = params[:p]
    betas = params[p:]

    circuit = build_qaoa_circuit(gammas, betas)
    result = sim.simulate(circuit)

    state = result.final_state_vector

    return compute_energy(state)

In [10]:
# inicialización aleatoria, se puede cambiar para ver mejora
init_params = np.random.uniform(0, np.pi, 2*p)

res = minimize(
    objective,
    init_params,
    method='COBYLA',   # robusto para QAOA
    options={'maxiter': 100}
)

print("Energía mínima encontrada:", res.fun)
print("Parámetros óptimos:", res.x)

Energía mínima encontrada: -12.244315147399902
Parámetros óptimos: [3.98176971 2.32035553 0.95716941 3.1894     1.71188484 2.72759513
 0.25289629 2.04292977 0.14946517 0.27337054 0.88817896 0.3463677 ]


In [13]:
def exact_classical_energy(n, J=1.0, h=0.0):
    best = 1e9
    for i in range(2**n):
        z = [(1 if ((i >> k) & 1) == 0 else -1) for k in range(n)]

        E = 0
        for j in range(n-1):
            E += -J * z[j] * z[j+1]

        for j in range(n):
            E += -h * z[j]

        best = min(best, E)

    return best
print("Exact:", exact_classical_energy(12, J, h))

Exact: -17.0
